In [ ]:
import pathlib

In [ ]:
import downstream
import imageio
import marimo as mo
import numpy as np
import polars as pl
import seaborn as sns
from teeplot import teeplot as tp
from tqdm.auto import tqdm
from watermark import watermark

from pylib.munge import addend_groups, squeeze_int, union_upsample

In [ ]:
mo.md(
    f"""
```Text
{watermark(
    current_date=True,
    iso8601=True,
    machine=True,
    updated=True,
    python=True,
    iversions=True,
    globals_=globals(),
)}
```
"""
)

```Text
Last updated: 2026-11-21T01:51:10.002465+00:00

Python implementation: CPython
Python version       : 3.10.12
IPython version      : 7.31.1

Compiler    : GCC 11.4.0
OS          : Linux
Release     : 6.8.0-1070-azure
Machine     : x86_64
Processor   : x86_64
CPU cores   : 4
Architecture: 64bit

teeplot   : 1.4.2
imageio   : 2.37.0
numpy     : 2.1.2
polars    : 1.40.1
downstream: 1.25.0
marimo    : 0.23.2
seaborn   : 0.13.2

```

# 2026-11-15-frozenseeded-fullgrid-p2p-trait-log

Spatial timeseries of logged trait prevalence for the 2026-11-15
frozen-seeded full-grid run with point-to-point immigrant exchange
(wse-async-ga trait logger data, OSF `jwuz7`).
Adapted from `binder/wse-denovo-spatial2d-explicitsite-timeseries.ipynb`
in mmore500/hypermutator-dynamics.

In [ ]:
nbname = pathlib.Path(__file__).stem
outdir = pathlib.Path("outdata") / nbname
outdir.mkdir(parents=True, exist_ok=True)
outdir

PosixPath('outdata/2026-11-15-frozenseeded-fullgrid-p2p-trait-log')

## Prep Data

In [ ]:
dfxs = pl.read_parquet("https://osf.io/jwuz7/download").with_row_index(
    "dstream_data_id",
)
dfxs.describe()

statistic,dstream_data_id,tile,row,col,genomeFlavor,globalSeed,dispatchSlug,populationFlavor,memcpyMajor,nCol,nRow,nWav,nTrait,nCycle,nColSubgrid,nRowSubgrid,nonBlock,tilePopSize,tournSize,msec,tsc,replicate,_compconf_dummy,CEREBRASLIB_TRAITLOGGER_NUM_BITS,CEREBRASLIB_POPULATION_EXTINCTION_PROBABILITY,ASYNC_GA_P2P_SEND_PROBABILITY,CEREBRASLIB_TRAITLOGGER_DILATION,CEREBRASLIB_TRAITLOGGER_DSTREAM_ALGO_NAME,ASYNC_GA_FOSSIL_SAMPLING_MODE,ASYNC_GA_GENOME_FLAVOR,ASYNC_GA_POPULATION_FLAVOR,ASYNC_GA_POPSIZE,data_hex,dstream_algo,dstream_storage_bitoffset,dstream_storage_bitwidth,dstream_S,dstream_T_bitoffset,dstream_T_bitwidth,trait_value
str,f64,f64,f64,f64,str,f64,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,str,str,str,str,f64,str,str,f64,f64,f64,f64,f64,f64
"""count""",883350.0,883350.0,883350.0,883350.0,"""883350""",883350.0,"""883350""","""883350""","""883350""",883350.0,883350.0,883350.0,883350.0,883350.0,883350.0,883350.0,883350.0,883350.0,883350.0,883350.0,883350.0,"""883350""",883350.0,883350.0,883350.0,883350.0,883350.0,"""883350""","""883350""","""883350""","""883350""",883350.0,"""883350""","""883350""",883350.0,883350.0,883350.0,883350.0,883350.0,883350.0
"""null_count""",0.0,0.0,0.0,0.0,"""0""",0.0,"""0""","""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""0""",0.0,0.0,0.0,0.0,0.0,"""0""","""0""","""0""","""0""",0.0,"""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",441674.5,441674.5,584.5,377.0,null,1115.0,null,null,null,755.0,1170.0,1.0,2.0,50000.0,0.0,0.0,0.0,1133.0,2.0,0.0,0.0,null,42.0,1024.0,0.0,0.0005,8.0,null,null,null,null,1133.0,null,null,0.0,1024.0,1024.0,1024.0,32.0,0.0
"""std""",255001.324482,255001.324482,337.749975,217.949659,null,0.0,null,null,null,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,0.0,0.0,0.0,0.0,null,0.0,0.0,0.0,0.0,0.0,null,null,null,null,0.0,null,null,0.0,0.0,0.0,0.0,0.0,0.0
"""min""",0.0,0.0,0.0,0.0,null,1115.0,null,null,null,755.0,1170.0,1.0,2.0,50000.0,0.0,0.0,0.0,1133.0,2.0,0.0,0.0,null,42.0,1024.0,0.0,0.0005,8.0,null,null,null,null,1133.0,"""c3fbfbefffffffffffffffffffffff…",null,0.0,1024.0,1024.0,1024.0,32.0,0.0
"""25%""",220837.0,220837.0,292.0,188.0,null,1115.0,null,null,null,755.0,1170.0,1.0,2.0,50000.0,0.0,0.0,null,1133.0,2.0,0.0,0.0,null,42.0,1024.0,0.0,0.0005,8.0,null,null,null,null,1133.0,null,null,0.0,1024.0,1024.0,1024.0,32.0,0.0
"""50%""",441675.0,441675.0,585.0,377.0,null,1115.0,null,null,null,755.0,1170.0,1.0,2.0,50000.0,0.0,0.0,null,1133.0,2.0,0.0,0.0,null,42.0,1024.0,0.0,0.0005,8.0,null,null,null,null,1133.0,null,null,0.0,1024.0,1024.0,1024.0,32.0,0.0
"""75%""",662512.0,662512.0,877.0,566.0,null,1115.0,null,null,null,755.0,1170.0,1.0,2.0,50000.0,0.0,0.0,null,1133.0,2.0,0.0,0.0,null,42.0,1024.0,0.0,0.0005,8.0,null,null,null,null,1133.0,null,null,0.0,1024.0,1024.0,1024.0,32.0,0.0
"""max""",883349.0,883349.0,1169.0,754.0,null,1115.0,null,null,null,755.0,1170.0,1.0,2.0,50000.0,0.0,0.0,0.0,1133.0,2.0,0.0,0.0,null,42.0,1024.0,0.0,0.0005,8.0,null,null,null,null,1133.0,"""ffffffffffffffffffffffffffffff…",null,0.0,1024.0,1024.0,1024.0,32.0,0.0


In [ ]:
def unpack_logger_timeseries_data(df: pl.DataFrame) -> pl.DataFrame:

    # sample every 64th tile
    df = df.filter((pl.col("col") % 8 == 0) & (pl.col("row") % 8 == 0))

    # unpack dstream data items from logger hex strings
    dfu = downstream.dataframe.unpack_data_packed(df)
    df = downstream.dataframe.explode_lookup_unpacked(
        dfu.with_columns(
            # fast forward loggers to at least dstream_S entries
            # (i.e., buffer filled)
            # these will be semantically-correct zero values
            dstream_T=pl.max_horizontal(
                pl.col("dstream_T"), pl.col("dstream_S")
            ),
        ),
        value_type="uint8",
    )

    # join metadata onto unpacked data
    df = df.join(
        dfu.select(
            "dstream_data_id",
            "CEREBRASLIB_TRAITLOGGER_DILATION",
            "col",
            "replicate",
            "row",
            "tile",
        ),
        on="dstream_data_id",
    )

    # apply dilation to dstream_T and dstream_Tbar
    df = df.with_columns(
        dstream_T=pl.col("dstream_T")
        * pl.col("CEREBRASLIB_TRAITLOGGER_DILATION"),
        dstream_Tbar=pl.col("dstream_Tbar")
        * pl.col("CEREBRASLIB_TRAITLOGGER_DILATION"),
    )

    # addend time time series groups with final zero value entry
    df = addend_groups(
        df,
        group_by=["replicate", "tile"],
        aggs=(
            pl.col("col").unique().pipe(squeeze_int),
            (pl.col("dstream_Tbar") + 1).max().alias("dstream_Tbar"),
            pl.lit(0).alias("dstream_value"),
            pl.col("row").unique().pipe(squeeze_int),
        ),
        inner_only=True,
    )

    # upsample to fill in dstream_Tbar dropped by some loggers
    # prevents sampling biases from longer- vs shorter-running loggers
    df = union_upsample(
        df,
        upsample="dstream_Tbar",
        group_by="tile",
        fill_null_ops=[
            (pl.all(), {"strategy": "forward"}),
            (pl.col("dstream_Tbar", "dstream_value"), {"strategy": "zero"}),
            (pl.all(), {"strategy": "backward"}),
        ],
    )
    assert len(df["dstream_Tbar"].value_counts()["count"].unique()) == 1

    return df

In [ ]:
df = pl.concat(
    [
        unpack_logger_timeseries_data(_df)
        for _, _df in tqdm(dfxs.group_by("replicate"))
    ],
)
df

0it [00:00, ?it/s]/usr/local/lib/python3.10/dist-packages/downstream/dataframe/_impl/_check_downstream_version.py:34: UserWarning: Dataframe downstream_version column not provided
  warnings.warn(
1it [00:30, 30.28s/it]


tile,dstream_Tbar,replicate,col,dstream_value,row
u32,u64,cat,i64,i32,i64
416856,8,"""fb669295-10be-4423-b7d7-c3d5d6…",96,0,552
416856,24,"""fb669295-10be-4423-b7d7-c3d5d6…",96,0,552
416856,40,"""fb669295-10be-4423-b7d7-c3d5d6…",96,0,552
416856,48,"""fb669295-10be-4423-b7d7-c3d5d6…",96,0,552
416856,56,"""fb669295-10be-4423-b7d7-c3d5d6…",96,0,552
…,…,…,…,…,…
713072,49648,"""fb669295-10be-4423-b7d7-c3d5d6…",352,1,944
713072,49656,"""fb669295-10be-4423-b7d7-c3d5d6…",352,1,944
713072,49776,"""fb669295-10be-4423-b7d7-c3d5d6…",352,1,944


## Plot Data

In [ ]:
# take the mean of dstream_value at each dstream_Tbar across tiles
df_mean = df.group_by("replicate", "dstream_Tbar").agg(
    pl.col("dstream_value").mean(),
)

In [ ]:
with tp.teed(
    sns.relplot,
    data=df_mean.sort("dstream_Tbar"),
    x="dstream_Tbar",
    y="dstream_value",
    hue="replicate",
    alpha=0.5,
    aspect=1.5,
    errorbar=None,
    height=2.5,
    linewidth=1,
    legend=False,
    kind="line",
    marker="d",
    markersize=4,
    teeplot_subdir=nbname,
    teeplot_outexclude=["marker"],
) as _g:
    _g.set(xlim=(10, 80000))
    _g.set(ylim=(0.00001, 2))
    _g.set_axis_labels(x_var="Generations", y_var="Trait\nPrevalence")
    for _ax in _g.axes.flat:
        _ax.axhline(1, ls="--", color="black", linewidth=1, zorder=-10)
        _ax.set_yscale("log")
        _ax.set_xscale("log")

teeplots/2026-11-15-frozenseeded-fullgrid-p2p-trait-log/hue=replicate+kind=line+viz=relplot+x=dstream-tbar+y=dstream-value+ext=.pdf
teeplots/2026-11-15-frozenseeded-fullgrid-p2p-trait-log/hue=replicate+kind=line+viz=relplot+x=dstream-tbar+y=dstream-value+ext=.png


## Animate Data

In [ ]:
for (_replicate,), _dfr in tqdm(df.group_by("replicate")):
    _frames = []
    for (_dstream_Tbar,), _dff in _dfr.sort("dstream_Tbar").group_by(
        "dstream_Tbar", maintain_order=True
    ):
        _dff = _dff.with_columns(
            col_rank=pl.col("col").rank(method="dense").cast(pl.Int32) - 1,
            row_rank=pl.col("row").rank(method="dense").cast(pl.Int32) - 1,
        )

        _num_rows = _dff["row_rank"].max() + 1
        _num_cols = _dff["col_rank"].max() + 1

        _frame = np.empty((_num_rows, _num_cols), dtype=np.uint8)
        _frame[_dff["row_rank"], _dff["col_rank"]] = (
            _dff["dstream_value"] * 255
        )
        _frame = np.kron(_frame, np.ones((4, 4), dtype=_frame.dtype))
        _frames.append((_dstream_Tbar, _frame))

        _framedir = outdir / f"a=traitframes+rep={_replicate}"
        _framedir.mkdir(parents=True, exist_ok=True)
        imageio.imwrite(
            _framedir / f"dstream_Tbar={_dstream_Tbar:06}+ext=.png",
            _frame.T,
        )

    print(len(_frames))
    imageio.mimsave(
        outdir / f"a=traits+rep={_replicate}+ext=.gif",
        [_f for _, _f in _frames],
        duration=0.2,
    )

    # sweep completes within first ~1000 generations; animate that window
    _sweep = [_f.T for _t, _f in _frames if _t <= 1000]
    print(len(_sweep))
    imageio.mimsave(
        outdir / f"a=traits-sweep+rep={_replicate}+tbar_max=1000+ext=.gif",
        _sweep,
        duration=0.1,
        loop=0,
    )

0it [00:00, ?it/s]

1025
74


1it [00:09,  9.04s/it]
